# Rohbau Annotator Demo

**rohbau-annotator** is an annotation tool for [Rohbau3D](https://github.com/2Dpass/rohbau3d) construction-site point clouds.
It works by projecting a 3D point cloud onto a 2D equirectangular (spherical) panorama, letting you annotate in 2D, and then back-projecting labels to 3D.

## Rohbau3D data format

Each scan directory contains:

| File | Shape | Description |
|---|---|---|
| `coord.npy` | `(N, 3) float64` | XYZ point coordinates |
| `color.npy` | `(N, 3) uint8` | RGB point colors |
| `intensity.npy` | `(N, 1) float64` | Laser return intensity |
| `normal.npy` | `(N, 3) float64` | Surface normals |
| `img_color.png` | `(H, W, 3) uint8` | Panorama color image |

## SAM2 integration

The tool optionally integrates **Segment Anything Model 2 (SAM2)** for semi-automatic annotation.
When SAM2 mode is enabled, left-clicks place foreground prompts and right-clicks place background prompts.
SAM2 proposes a mask in real time; press Enter to accept or Escape to reject.

## What this notebook covers

1. Create synthetic point-cloud data for a simple room
2. Demonstrate the spherical projection round-trip (3D -> 2D -> 3D)
3. Render a panorama image from the point cloud
4. Show the 13 semantic class definitions with colors
5. Simulate a 2D annotation and back-project to 3D
6. Visualize the results with matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from rohbau_annotator.projection import (
    points_to_panorama_uv,
    points_to_panorama_indices,
    panorama_uv_to_points,
    build_panorama_label_map,
    render_label_panorama,
)
from rohbau_annotator.classes import CLASSES, CLASS_BY_ID, class_color_norm

%matplotlib inline

## 1. Create synthetic scan data

We generate a simple box-shaped room (5 m x 4 m x 3 m) centered at the origin.
The scanner position is at `(0, 0, 0)` -- the center of the room.
Each surface (floor, ceiling, four walls) gets a distinct color so we can
verify the projection visually.

In [ ]:
rng = np.random.default_rng(42)

def make_wall_points(axis, value, bounds_a, bounds_b, n, height_axis, width_axis):
    """Generate points on an axis-aligned plane."""
    pts = np.zeros((n, 3))
    pts[:, height_axis] = rng.uniform(*bounds_a, size=n)
    pts[:, width_axis] = rng.uniform(*bounds_b, size=n)
    pts[:, axis] = value
    return pts

N_PER_FACE = 5000

# Room dimensions: x in [-2.5, 2.5], y in [-2, 2], z in [-1.5, 1.5]
floor   = make_wall_points(2, -1.5, (-2.5, 2.5), (-2, 2), N_PER_FACE, 0, 1)
ceiling = make_wall_points(2,  1.5, (-2.5, 2.5), (-2, 2), N_PER_FACE, 0, 1)
wall_px = make_wall_points(0,  2.5, (-2, 2), (-1.5, 1.5), N_PER_FACE, 1, 2)  # +X wall
wall_nx = make_wall_points(0, -2.5, (-2, 2), (-1.5, 1.5), N_PER_FACE, 1, 2)  # -X wall
wall_py = make_wall_points(1,  2.0, (-2.5, 2.5), (-1.5, 1.5), N_PER_FACE, 0, 2)  # +Y wall
wall_ny = make_wall_points(1, -2.0, (-2.5, 2.5), (-1.5, 1.5), N_PER_FACE, 0, 2)  # -Y wall

coord = np.vstack([floor, ceiling, wall_px, wall_nx, wall_py, wall_ny]).astype(np.float64)

# Assign colors per surface (RGB 0-255)
colors = {
    'floor':   [152, 223, 138],  # green
    'ceiling': [ 31, 119, 180],  # blue
    'wall_px': [174, 199, 232],  # light blue
    'wall_nx': [214,  39,  40],  # red
    'wall_py': [255, 187, 120],  # orange
    'wall_ny': [148, 103, 189],  # purple
}
color = np.vstack([
    np.tile(colors['floor'],   (N_PER_FACE, 1)),
    np.tile(colors['ceiling'], (N_PER_FACE, 1)),
    np.tile(colors['wall_px'], (N_PER_FACE, 1)),
    np.tile(colors['wall_nx'], (N_PER_FACE, 1)),
    np.tile(colors['wall_py'], (N_PER_FACE, 1)),
    np.tile(colors['wall_ny'], (N_PER_FACE, 1)),
]).astype(np.uint8)

# Synthetic intensity and normals (not used in this demo, but part of the format)
intensity = rng.uniform(0.3, 1.0, size=(coord.shape[0], 1))
normal = np.zeros_like(coord)  # placeholder

print(f"Point cloud: {coord.shape[0]} points")
print(f"  coord   : {coord.shape} {coord.dtype}")
print(f"  color   : {color.shape} {color.dtype}")
print(f"  intensity: {intensity.shape}")

## 2. Spherical projection round-trip

The core of rohbau-annotator is the equirectangular spherical projection.

- **`points_to_panorama_uv`** maps 3D `(x, y, z)` to floating-point panorama coordinates `(u, v)` plus radial depth `r`.
- **`panorama_uv_to_points`** reconstructs 3D coordinates from `(u, v, depth)`.

A perfect round-trip should recover the original points exactly (up to floating-point precision).

In [ ]:
IMG_H, IMG_W = 512, 1024

# Forward: 3D -> (u, v, r)
u, v, r = points_to_panorama_uv(coord, IMG_H, IMG_W)

print(f"u range: [{u.min():.2f}, {u.max():.2f}]  (expected [0, {IMG_W}])")
print(f"v range: [{v.min():.2f}, {v.max():.2f}]  (expected [0, {IMG_H}])")
print(f"r range: [{r.min():.2f}, {r.max():.2f}] meters")

# Backward: (u, v, r) -> 3D
reconstructed = panorama_uv_to_points(u, v, r, IMG_H, IMG_W)

error = np.linalg.norm(coord - reconstructed, axis=1)
print(f"\nRound-trip reconstruction error:")
print(f"  mean  : {error.mean():.2e} m")
print(f"  max   : {error.max():.2e} m")
print(f"  median: {np.median(error):.2e} m")

assert error.max() < 1e-10, "Round-trip error too large!"
print("\nRound-trip accuracy verified: all errors < 1e-10 m")

## 3. Render a panorama image from the point cloud

We project every 3D point onto the panorama grid and use z-buffering
(closest point wins) to produce an `(H, W, 3)` color image.

In [ ]:
def render_color_panorama(points, colors_rgb, img_h, img_w):
    """Render a color panorama with z-buffering."""
    rows, cols, r = points_to_panorama_indices(points, img_h, img_w)

    panorama = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    depth_buf = np.full((img_h, img_w), np.inf)

    for i in range(len(points)):
        row, col, dist = rows[i], cols[i], r[i]
        if dist < depth_buf[row, col]:
            depth_buf[row, col] = dist
            panorama[row, col] = colors_rgb[i]

    return panorama, depth_buf

panorama, depth_buf = render_color_panorama(coord, color, IMG_H, IMG_W)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].imshow(panorama)
axes[0].set_title("Panorama - RGB color")
axes[0].set_ylabel("v (row)")
axes[0].set_xlabel("u (column)")

depth_display = depth_buf.copy()
depth_display[depth_display == np.inf] = np.nan
im = axes[1].imshow(depth_display, cmap="viridis_r")
axes[1].set_title("Panorama - Depth")
axes[1].set_ylabel("v (row)")
axes[1].set_xlabel("u (column)")
plt.colorbar(im, ax=axes[1], label="depth (m)")

plt.tight_layout()
plt.show()

## 4. Semantic class definitions

Rohbau-annotator defines **13 semantic classes** for construction-site elements
(plus class 0 = unlabeled). Each class has an integer ID, a name, and an RGB color.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.set_xlim(0, 1)
ax.set_ylim(0, len(CLASSES))
ax.set_axis_off()
ax.set_title("Rohbau3D Semantic Classes", fontsize=13, fontweight="bold")

for i, cls in enumerate(CLASSES):
    y = len(CLASSES) - 1 - i
    rgb_norm = class_color_norm(cls)
    ax.barh(y, 0.3, color=rgb_norm, height=0.8, left=0.0, edgecolor="gray", linewidth=0.5)
    ax.text(0.35, y, f"{cls.id:2d}  {cls.name}", va="center", fontsize=10, fontfamily="monospace")
    ax.text(0.75, y, f"RGB({cls.color[0]:3d},{cls.color[1]:3d},{cls.color[2]:3d})",
            va="center", fontsize=8, fontfamily="monospace", color="gray")

plt.tight_layout()
plt.show()

## 5. Simulate annotation and back-project to 3D

In a real workflow, the user paints labels on the panorama interactively with
`PanoramaAnnotator.run()`. Here we simulate that by creating a 2D label map
programmatically, then use `build_panorama_label_map` to transfer labels to 3D.

We assign:
- **floor** (class 2) to the bottom strip of the panorama (high v = low elevation)
- **ceiling** (class 3) to the top strip
- **wall** (class 1) to the middle band

In [ ]:
# Create a simulated 2D label map
label_2d = np.zeros((IMG_H, IMG_W), dtype=np.int32)

# Ceiling region: top ~25% of panorama (low v = looking up)
label_2d[:int(IMG_H * 0.25), :] = 3   # ceiling

# Floor region: bottom ~25% of panorama (high v = looking down)
label_2d[int(IMG_H * 0.75):, :] = 2   # floor

# Wall region: middle band
label_2d[int(IMG_H * 0.25):int(IMG_H * 0.75), :] = 1  # wall

# Add a "door" rectangle on the wall region
door_v_start = int(IMG_H * 0.35)
door_v_end   = int(IMG_H * 0.70)
door_u_start = int(IMG_W * 0.10)
door_u_end   = int(IMG_W * 0.18)
label_2d[door_v_start:door_v_end, door_u_start:door_u_end] = 6  # door

# Add a "window" rectangle
win_v_start = int(IMG_H * 0.30)
win_v_end   = int(IMG_H * 0.55)
win_u_start = int(IMG_W * 0.55)
win_u_end   = int(IMG_W * 0.70)
label_2d[win_v_start:win_v_end, win_u_start:win_u_end] = 7  # window

# Back-project to 3D
labels_3d = build_panorama_label_map(label_2d, coord, IMG_H, IMG_W)

# Statistics
print("Label statistics (3D points):")
print(f"{'Class':>15s}  {'ID':>3s}  {'Count':>7s}  {'Pct':>6s}")
print("-" * 38)
for cls in CLASSES:
    count = int((labels_3d == cls.id).sum())
    if count > 0:
        pct = 100.0 * count / len(labels_3d)
        print(f"{cls.name:>15s}  {cls.id:3d}  {count:7d}  {pct:5.1f}%")

print(f"\nTotal points: {len(labels_3d)}")
print(f"Labeled:      {(labels_3d > 0).sum()} ({100*(labels_3d>0).mean():.1f}%)")

## 6. Visualization

We create two views:
1. **Panorama** with the simulated annotation overlay
2. **3D scatter plot** of the point cloud colored by semantic class

In [ ]:
# Build a color map for the label panorama
def label_map_to_rgb(label_map):
    """Convert an integer label map to an RGB image."""
    h, w = label_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls in CLASSES:
        mask = label_map == cls.id
        if mask.any():
            rgb[mask] = cls.color
    return rgb

label_rgb_2d = label_map_to_rgb(label_2d)

# Panorama visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].imshow(panorama)
axes[0].set_title("Original panorama")
axes[0].set_axis_off()

# Blend panorama with label overlay
blended = (0.5 * panorama.astype(float) + 0.5 * label_rgb_2d.astype(float)).astype(np.uint8)
axes[1].imshow(blended)
axes[1].set_title("Panorama + annotation overlay")
axes[1].set_axis_off()

# Legend
used_ids = np.unique(label_2d)
legend_patches = [
    Patch(facecolor=class_color_norm(CLASS_BY_ID[cid]), label=CLASS_BY_ID[cid].name)
    for cid in used_ids if cid > 0
]
axes[1].legend(handles=legend_patches, loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# 3D scatter plot colored by semantic class
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")

# Map each point to its class color
point_colors = np.zeros((len(coord), 3), dtype=np.float64)
for cls in CLASSES:
    mask = labels_3d == cls.id
    if mask.any():
        point_colors[mask] = [c / 255.0 for c in cls.color]

# Subsample for faster rendering
subsample = 4
idx = np.arange(0, len(coord), subsample)

ax.scatter(
    coord[idx, 0], coord[idx, 1], coord[idx, 2],
    c=point_colors[idx],
    s=0.5,
    alpha=0.6,
)

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.set_title("3D point cloud colored by semantic class")

# Legend
legend_patches = [
    Patch(facecolor=class_color_norm(CLASS_BY_ID[cid]), label=CLASS_BY_ID[cid].name)
    for cid in np.unique(labels_3d) if cid > 0
]
ax.legend(handles=legend_patches, loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

print("Demo complete.")